# Minimalist QnA Bot (LangChain + Ollama)

ReAct agent answering questions about ingredients, menu, and transactions using a local Ollama model. The agent picks which dataset tool to call per question instead of stuffing all data into the prompt.

In [1]:
import json
from pathlib import Path

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

DATA_DIR = Path("../data")
MODEL = "qwen3.5:4b"

In [2]:
def _load(filename: str) -> str:
    return json.dumps(json.loads((DATA_DIR / filename).read_text()))


@tool
def get_ingredients() -> str:
    """Return current stock inventory: ingredient id, name, amount, unit, purchase date."""
    return _load("ingredients.json")


@tool
def get_menu() -> str:
    """Return menu items: id, name, description, price, availability."""
    return _load("menu.json")


@tool
def get_transactions() -> str:
    """Return sales transactions: id, timestamp, status, total, line items."""
    return _load("transactions.json")


tools = [get_ingredients, get_menu, get_transactions]

In [3]:
SYSTEM_PROMPT = (
    "You answer questions about a restaurant's stock inventory, menu, and "
    "transactions. Call the relevant tool(s) to fetch data before answering. "
    "If the answer isn't in the data, say so."
)

llm = ChatOllama(model=MODEL, temperature=0)
agent = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)


def ask(question: str) -> str:
    result = agent.invoke({"messages": [("human", question)]})
    return result["messages"][-1].content

In [4]:
print(ask("What is the best selling menu?"))

Based on the transaction data available, the **Margherita Pizza** is the best-selling menu item with 2 units sold across completed transactions.

Here's a breakdown of sales from completed transactions:
- **Classic Cheeseburger**: 1 unit
- **Iced Caramel Macchiato**: 1 unit
- **Margherita Pizza**: 2 units (best seller)
- **Mango Passionfruit Smoothie**: 2 units
- **Chocolate Lava Cake**: 1 unit

Note: One transaction (TXN-10003) was cancelled and doesn't count toward sales totals. This analysis is based on the limited transaction data available - there may be additional transactions not shown here.


In [5]:
while True:
    q = input("Ask (blank to quit): ").strip()
    if not q:
        break
    print(ask(q), "\n")